<a href="https://www.kaggle.com/code/halaelbayoumi/hidden-cost-of-ml-projects-resnet-18-on-cifar-10?scriptVersionId=304137628" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
#  ENVIRONMENT SETUP
# ============================================================
!pip install codecarbon -q

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import time
import psutil
import os

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score
from codecarbon import EmissionsTracker

# --- Check GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


#  DATA PIPELINE
# ============================================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --- Load CIFAR-10 ---
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_dataset       = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# --- Split train into train and validation (80/20) ---
train_size = int(0.8 * len(full_train_dataset))
val_size   = len(full_train_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(full_train_dataset, [train_size, val_size])

# --- Data Loaders ---
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size  = 32,
    shuffle     = True,
    num_workers = 4,
    pin_memory  = True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True
)

print(f"Training samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Testing samples   : {len(test_dataset)}")


#  MODEL SETUP (VERIFICATION ONLY — LOOP REINITIALIZES)
# ============================================================
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(512, 10)
for param in model.parameters():
    param.requires_grad = True
model = model.to(device)

print(model.fc)
print(f"\nTotal parameters    : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


#  TRAINING FUNCTION WITH EARLY STOPPING
# ============================================================
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30, patience=5):

    best_val_loss  = float('inf')
    patience_count = 0
    best_weights   = None

    for epoch in range(epochs):

        # --- Training phase ---
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        # --- Validation phase ---
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        val_loss = val_loss / len(val_loader)

        print(f"  Epoch [{epoch+1}/{epochs}]  Train Loss: {epoch_loss:.4f}  Val Loss: {val_loss:.4f}")

        # --- Early stopping check ---
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_weights   = model.state_dict().copy()
            print(f"  ✓ Val loss improved to {best_val_loss:.4f} — saving best weights")
        else:
            patience_count += 1
            print(f"  ✗ No improvement — patience {patience_count}/{patience}")

            if patience_count >= patience:
                print(f"\n  Early stopping triggered at epoch {epoch+1}")
                break

    # --- Restore best weights before returning ---
    model.load_state_dict(best_weights)
    return model


#  EVALUATION FUNCTION
# ============================================================
def evaluate_model(model, test_loader):
    model.eval()

    all_predictions = []
    all_labels      = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy  = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_predictions, average='macro', zero_division=0)

    return accuracy, precision, recall


# 10 RUNS LOOP
# ============================================================
NUM_RUNS = 10
VARIANT  = "ResNet-18 on CIFAR-10 Baseline"
results  = []

for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  Starting Run {run} of {NUM_RUNS}")
    print(f"{'='*60}")

    # --- Fresh Model Every Run ---
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(512, 10)
    for param in model.parameters():
        param.requires_grad = True
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # ----------------------------------------------------------
    # TRAINING — Time + Memory + CodeCarbon
    # ----------------------------------------------------------
    torch.cuda.reset_peak_memory_stats(device)
    process = psutil.Process(os.getpid())

    train_tracker = EmissionsTracker(
        project_name       = "resnet18_baseline_train",
        output_file        = "/kaggle/working/codecarbon_train.csv",
        measure_power_secs = 10,
        tracking_mode      = "machine",
        log_level          = "warning"
    )
    train_tracker.start()
    train_start = time.time()

    model = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30, patience=5)

    train_end = time.time()
    train_tracker.stop()

    train_time_sec    = round(train_end - train_start, 4)
    gpu_peak_train_mb = round(torch.cuda.max_memory_allocated(device) / (1024 ** 2), 4)
    cpu_peak_train_mb = round(process.memory_info().rss / (1024 ** 2), 4)

    # ----------------------------------------------------------
    # EVALUATION — Time + Memory + CodeCarbon
    # ----------------------------------------------------------
    torch.cuda.reset_peak_memory_stats(device)

    eval_tracker = EmissionsTracker(
        project_name       = "resnet18_baseline_eval",
        output_file        = "/kaggle/working/codecarbon_eval.csv",
        measure_power_secs = 10,
        tracking_mode      = "machine",
        log_level          = "warning"
    )
    eval_tracker.start()
    eval_start = time.time()

    accuracy, precision, recall = evaluate_model(model, test_loader)

    eval_end = time.time()
    eval_tracker.stop()

    eval_time_sec    = round(eval_end - eval_start, 4)
    gpu_peak_eval_mb = round(torch.cuda.max_memory_allocated(device) / (1024 ** 2), 4)
    cpu_peak_eval_mb = round(process.memory_info().rss / (1024 ** 2), 4)

    # ----------------------------------------------------------
    # STORE PERFORMANCE METRICS FOR THIS RUN
    # ----------------------------------------------------------
    run_result = {
        "variant"           : VARIANT,
        "run"               : run,
        "train_time_sec"    : train_time_sec,
        "eval_time_sec"     : eval_time_sec,
        "cpu_peak_train_mb" : cpu_peak_train_mb,
        "gpu_peak_train_mb" : gpu_peak_train_mb,
        "cpu_peak_eval_mb"  : cpu_peak_eval_mb,
        "gpu_peak_eval_mb"  : gpu_peak_eval_mb,
        "test_accuracy"     : round(accuracy,  4),
        "test_precision"    : round(precision, 4),
        "test_recall"       : round(recall,    4),
    }

    results.append(run_result)

    # --- Progressive save after every run ---
    pd.DataFrame(results).to_csv("/kaggle/working/resnet18_cifar10_baseline_results.csv", index=False)

    print(f"\n  Run {run} Complete")
    print(f"  Train Time     : {train_time_sec}s")
    print(f"  Eval Time      : {eval_time_sec}s")
    print(f"  GPU Peak Train : {gpu_peak_train_mb} MB")
    print(f"  GPU Peak Eval  : {gpu_peak_eval_mb} MB")
    print(f"  Accuracy       : {accuracy:.4f}")
    print(f"  Precision      : {precision:.4f}")
    print(f"  Recall         : {recall:.4f}")


# COMPUTE MEAN ROW + EXPORT TO CSV
# ============================================================
df = pd.DataFrame(results)

# --- Compute Mean Row ---
numeric_cols        = df.drop(columns=["variant", "run"]).mean()
mean_row            = numeric_cols.to_dict()
mean_row["variant"] = VARIANT
mean_row["run"]     = "mean"

df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

# --- Reorder Columns ---
cols = [
    "variant",           "run",
    "train_time_sec",    "eval_time_sec",
    "cpu_peak_train_mb", "gpu_peak_train_mb",
    "cpu_peak_eval_mb",  "gpu_peak_eval_mb",
    "test_accuracy",     "test_precision",   "test_recall"
]
df = df[cols]

# --- Export Final CSV ---
csv_filename = "/kaggle/working/resnet18_cifar10_baseline_results.csv"
df.to_csv(csv_filename, index=False)

print(f"\n{'='*60}")
print(f"  All {NUM_RUNS} runs completed!")
print(f"  Files saved:")
print(f"  → resnet18_cifar10_baseline_results.csv  (accuracy, precision, recall, timing)")
print(f"  → codecarbon_train.csv                   (training energy + CO2)")
print(f"  → codecarbon_eval.csv                    (evaluation energy + CO2)")
print(f"{'='*60}")
print(df[["variant", "run", "test_accuracy", "gpu_peak_train_mb"]])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.7 MB/s eta 0:00:00
Using device: cuda


100%|██████████| 170M/170M [00:01<00:00, 103MB/s]


Training samples  : 40000
Validation samples: 10000
Testing samples   : 10000
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]
[codecarbon WARNING @ 00:27:46] Multiple instances of codecarbon are allowed to run at the same time.


Linear(in_features=512, out_features=10, bias=True)

Total parameters    : 11,181,642
Trainable parameters: 11,181,642

  Starting Run 1 of 10


...

  Epoch [9/30]  Train Loss: 0.0688  Val Loss: 0.4842
  ✗ No improvement — patience 5/5

  Early stopping triggered at epoch 9

  Run 9 Complete
  Train Time     : 1218.2973s
  Eval Time      : 9.948s
  GPU Peak Train : 898.584 MB
  GPU Peak Eval  : 420.3076 MB
  Accuracy       : 0.8760
  Precision      : 0.8813
  Recall         : 0.8760

  Starting Run 10 of 10


[codecarbon WARNING @ 03:42:12] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 03:42:12] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 03:42:12] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 03:42:12] No CPU tracking mode found. Falling back on CPU constant mode.


  Epoch [1/30]  Train Loss: 0.7248  Val Loss: 0.4936
  ✓ Val loss improved to 0.4936 — saving best weights
  Epoch [2/30]  Train Loss: 0.4332  Val Loss: 0.3933
  ✓ Val loss improved to 0.3933 — saving best weights
  Epoch [3/30]  Train Loss: 0.3081  Val Loss: 0.4088
  ✗ No improvement — patience 1/5
  Epoch [4/30]  Train Loss: 0.2331  Val Loss: 0.3392
  ✓ Val loss improved to 0.3392 — saving best weights
  Epoch [5/30]  Train Loss: 0.1644  Val Loss: 0.3406
  ✗ No improvement — patience 1/5
  Epoch [6/30]  Train Loss: 0.1333  Val Loss: 0.3950
  ✗ No improvement — patience 2/5
  Epoch [7/30]  Train Loss: 0.1038  Val Loss: 0.3738
  ✗ No improvement — patience 3/5
  Epoch [8/30]  Train Loss: 0.0860  Val Loss: 0.4132
  ✗ No improvement — patience 4/5


[codecarbon WARNING @ 04:02:33] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 04:02:33] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 04:02:33] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 04:02:33] No CPU tracking mode found. Falling back on CPU constant mode.


  Epoch [9/30]  Train Loss: 0.0804  Val Loss: 0.4003
  ✗ No improvement — patience 5/5

  Early stopping triggered at epoch 9

  Run 10 Complete
  Train Time     : 1217.8268s
  Eval Time      : 9.9191s
  GPU Peak Train : 897.459 MB
  GPU Peak Eval  : 418.1201 MB
  Accuracy       : 0.8887
  Precision      : 0.8922
  Recall         : 0.8887

  All 10 runs completed!
  Files saved:
  → resnet18_cifar10_baseline_results.csv  (accuracy, precision, recall, timing)
  → codecarbon_train.csv                   (training energy + CO2)
  → codecarbon_eval.csv                    (evaluation energy + CO2)
                           variant   run  test_accuracy  gpu_peak_train_mb
0   ResNet-18 on CIFAR-10 Baseline     1        0.88960          897.95900
1   ResNet-18 on CIFAR-10 Baseline     2        0.87700          897.89650
2   ResNet-18 on CIFAR-10 Baseline     3        0.88580          899.95900
3   ResNet-18 on CIFAR-10 Baseline     4        0.86440          899.27150
4   ResNet-18 on CIFAR-10 